# BMIN 5200 — Week 8 in-class exercise
## Bayesian networks you can interrogate, and machines with memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week08.ipynb)

**Time:** ~25 minutes · **Pairs with:** Bayesian networks; state machines

### What you'll do
- Build a six-node clinical Bayes net with a real causal story and fill in its conditional probability tables
- Run variable elimination to watch a diagnosis update as evidence arrives, one finding at a time
- Predict which conditional independences hold, write the predictions down, and check them against pgmpy's d-separation test
- Implement a heparin titration protocol as a finite-state machine, then run Conway's Game of Life in fifteen lines

### Why it matters
A Bayes net is the first representation in this course that lets you ask *why* a probability moved. Explaining away — where two independent causes become dependent the moment you observe their shared effect — is the formal version of what a clinician does when a positive result for one diagnosis makes them stop working up another, and it is the single most counterintuitive consequence of the graph structure. The second half is the other half of the week: a state machine is what you reach for when the same input means different things depending on where the patient already is, which is exactly how titration protocols and escalation pathways are written.

## Setup

`pgmpy` is not preinstalled in Colab; the install takes about twenty seconds. The second half of
the notebook needs nothing beyond the standard library.

In [ ]:
%pip install -q pgmpy
import warnings
warnings.filterwarnings("ignore")     # pgmpy emits deprecation notices we do not need in class

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

rng = np.random.default_rng(5200)

## Part 1 — A network with a causal story

Six variables, all binary, and every edge points from a cause to an effect. Smoking raises the
risk of both COPD and lung cancer. COPD and lung cancer both cause dyspnea, and so does heart
failure, which has nothing to do with smoking in this model. A chest X-ray is a test that reads
abnormal mostly when there is a cancer to see. The numbers below are plausible but invented;
treat them as a teaching model, not a risk calculator.

In [ ]:
clinical_network = DiscreteBayesianNetwork([
    ("smoking", "copd"),
    ("smoking", "lung_cancer"),
    ("lung_cancer", "chest_xray"),
    ("copd", "dyspnea"),
    ("lung_cancer", "dyspnea"),
    ("heart_failure", "dyspnea"),
])

node_positions = {
    "smoking": (0.0, 3.0),
    "copd": (-1.2, 2.0),
    "lung_cancer": (1.2, 2.0),
    "heart_failure": (-3.2, 1.0),
    "chest_xray": (2.6, 1.0),
    "dyspnea": (-0.6, 0.0),
}

plt.figure(figsize=(8, 5))
nx.draw(clinical_network, pos=node_positions, with_labels=True, node_size=4200,
        arrowsize=18, font_size=9)
plt.title("Every edge points from a cause to an effect")
plt.show()

print("parents of dyspnea:", sorted(clinical_network.get_parents("dyspnea")))
print("parents of chest_xray:", sorted(clinical_network.get_parents("chest_xray")))
print("parents of heart_failure:", sorted(clinical_network.get_parents("heart_failure")))

Now the conditional probability tables, which is the "Example: Local conditional probabilities"
slide. Each node needs one column of probabilities for every combination of its parents' values,
so `dyspnea` with three binary parents needs eight columns. The Markov assumption is what makes
this tractable: a node's distribution depends on its parents and nothing else, so specifying six
small tables is enough to determine the full joint over 2⁶ = 64 states.

In [ ]:
cpd_smoking = TabularCPD("smoking", 2, [[0.75], [0.25]],
                         state_names={"smoking": ["no", "yes"]})

cpd_heart_failure = TabularCPD("heart_failure", 2, [[0.92], [0.08]],
                               state_names={"heart_failure": ["no", "yes"]})

cpd_copd = TabularCPD("copd", 2,
                      [[0.98, 0.80],       # P(copd = no  | smoking = no / yes)
                       [0.02, 0.20]],      # P(copd = yes | smoking = no / yes)
                      evidence=["smoking"], evidence_card=[2],
                      state_names={"copd": ["no", "yes"], "smoking": ["no", "yes"]})

cpd_lung_cancer = TabularCPD("lung_cancer", 2,
                             [[0.998, 0.94],
                              [0.002, 0.06]],
                             evidence=["smoking"], evidence_card=[2],
                             state_names={"lung_cancer": ["no", "yes"], "smoking": ["no", "yes"]})

cpd_chest_xray = TabularCPD("chest_xray", 2,
                            [[0.95, 0.10],  # P(normal   | lung_cancer = no / yes)
                             [0.05, 0.90]], # P(abnormal | lung_cancer = no / yes)
                            evidence=["lung_cancer"], evidence_card=[2],
                            state_names={"chest_xray": ["normal", "abnormal"],
                                         "lung_cancer": ["no", "yes"]})

# Eight parent combinations, in the order (copd, lung_cancer, heart_failure) with the last
# variable changing fastest: (n,n,n) (n,n,y) (n,y,n) (n,y,y) (y,n,n) (y,n,y) (y,y,n) (y,y,y).
dyspnea_given_causes = [0.03, 0.86, 0.47, 0.92, 0.56, 0.94, 0.76, 0.96]

cpd_dyspnea = TabularCPD("dyspnea", 2,
                         [[round(1 - p, 2) for p in dyspnea_given_causes],
                          dyspnea_given_causes],
                         evidence=["copd", "lung_cancer", "heart_failure"], evidence_card=[2, 2, 2],
                         state_names={"dyspnea": ["no", "yes"], "copd": ["no", "yes"],
                                      "lung_cancer": ["no", "yes"], "heart_failure": ["no", "yes"]})

clinical_network.add_cpds(cpd_smoking, cpd_heart_failure, cpd_copd,
                          cpd_lung_cancer, cpd_chest_xray, cpd_dyspnea)

print("network is internally consistent:", clinical_network.check_model())
print()
print("P(dyspnea = yes | its three causes):")
print(f"  {'COPD':>5}  {'lung cancer':>12}  {'heart failure':>14}   P(dyspnea)")
column = 0
for copd_state in ("no", "yes"):
    for cancer_state in ("no", "yes"):
        for failure_state in ("no", "yes"):
            print(f"  {copd_state:>5}  {cancer_state:>12}  {failure_state:>14}"
                  f"   {dyspnea_given_causes[column]:.2f}")
            column += 1

Read the last column of that table down. With no cause present the chance of dyspnea is 3%,
which is the background rate of breathlessness from everything we did not model. Heart failure
alone takes it to 86%, COPD alone to 56%, lung cancer alone to 47%, and all three together to
96%. Nothing in the table says the three causes are related to each other — that information
lives entirely in the graph.

## Part 2 — Asking the network questions

Variable elimination computes a marginal by summing the other variables out of the joint in a
sensible order. In practice you hand pgmpy a query variable and some evidence and read the answer.
Start with a patient about whom we know nothing, then add findings one at a time and watch the
posterior for lung cancer move.

In [ ]:
inference = VariableElimination(clinical_network)


def probability(variable, evidence=None, state="yes"):
    """P(variable = state | evidence). Evidence is a dict of variable -> state name."""
    result = inference.query(variables=[variable], evidence=evidence, show_progress=False)
    return result.get_value(**{variable: state})


print("Posterior probability of lung cancer as findings accumulate:")
print(f"  {'evidence':<52}  P(lung cancer)")
for label, evidence in [
    ("nothing known", None),
    ("dyspnea", {"dyspnea": "yes"}),
    ("dyspnea, abnormal chest x-ray", {"dyspnea": "yes", "chest_xray": "abnormal"}),
    ("dyspnea, abnormal chest x-ray, smoker", {"dyspnea": "yes", "chest_xray": "abnormal",
                                               "smoking": "yes"}),
]:
    print(f"  {label:<52}  {probability('lung_cancer', evidence):.4f}")

### Predict before you run

A patient presents with dyspnea. Before running the next cell, commit to answers out loud.

1. `heart_failure` and `lung_cancer` have no edge between them and no shared parent. Before any
   evidence arrives, are they independent?
2. The patient has dyspnea. Does the probability of heart failure go up?
3. A chest CT then confirms lung cancer, which fully explains the dyspnea. What happens to the
   probability of heart failure — up, down, or unchanged? Say which, and by roughly how much.

Write your three answers down before you run anything.

In [ ]:
print("Heart failure, as evidence about a completely different disease arrives:")
print(f"  {'what we know':<48}  P(heart failure)")
for label, evidence in [
    ("nothing", None),
    ("the patient has dyspnea", {"dyspnea": "yes"}),
    ("dyspnea, and lung cancer is confirmed", {"dyspnea": "yes", "lung_cancer": "yes"}),
    ("dyspnea, and COPD is confirmed instead", {"dyspnea": "yes", "copd": "yes"}),
]:
    print(f"  {label:<48}  {probability('heart_failure', evidence):.4f}")

print()
print("And the same thing in the other direction:")
print(f"  {'what we know':<48}  P(lung cancer)")
for label, evidence in [
    ("nothing", None),
    ("the patient has dyspnea", {"dyspnea": "yes"}),
    ("dyspnea, and heart failure is confirmed", {"dyspnea": "yes", "heart_failure": "yes"}),
]:
    print(f"  {label:<48}  {probability('lung_cancer', evidence):.4f}")

This is explaining away. Heart failure starts at 8%, and dyspnea raises it to 51% because
heart failure is one of the few things in this model that causes dyspnea. Then lung cancer is
confirmed, and heart failure falls back to 13% — not because anything about the heart changed, but
because the symptom that was raising suspicion now has an owner. The two causes compete for the
same evidence.

Note what this means structurally: `heart_failure` and `lung_cancer` are independent in the
network as drawn, and they became dependent *because* we observed their shared effect.
Conditioning on evidence usually removes dependence. At a collider it creates it.

### Your turn: predict seven independence questions, then check them

The Bayes-ball rules from the slides let you answer these from the picture alone, with no
arithmetic. For each pair below, decide whether the two variables are **dependent** or
**independent** given what is observed. Write your seven answers into the dictionary in the next
cell before running it — the placeholder answers "dependent" to everything, which is wrong for
some of them.

| # | question |
|---|----------|
| 1 | `chest_xray` and `smoking`, nothing observed |
| 2 | `chest_xray` and `smoking`, having observed `lung_cancer` |
| 3 | `heart_failure` and `lung_cancer`, nothing observed |
| 4 | `heart_failure` and `lung_cancer`, having observed `dyspnea` |
| 5 | `copd` and `lung_cancer`, nothing observed |
| 6 | `copd` and `lung_cancer`, having observed `smoking` |
| 7 | `heart_failure` and `chest_xray`, having observed `dyspnea` |

In [ ]:
INDEPENDENCE_QUESTIONS = [
    (1, "chest_xray", "smoking", []),
    (2, "chest_xray", "smoking", ["lung_cancer"]),
    (3, "heart_failure", "lung_cancer", []),
    (4, "heart_failure", "lung_cancer", ["dyspnea"]),
    (5, "copd", "lung_cancer", []),
    (6, "copd", "lung_cancer", ["smoking"]),
    (7, "heart_failure", "chest_xray", ["dyspnea"]),
]

# TODO: replace each "dependent" with your own answer: "dependent" or "independent".
MY_PREDICTIONS = {
    1: "dependent",
    2: "dependent",
    3: "dependent",
    4: "dependent",
    5: "dependent",
    6: "dependent",
    7: "dependent",
}

print(f"  {'#':>2}  {'pair':<32}  {'observed':<16}  {'you said':<12}  {'graph says':<12}")
correct = 0
for number, first, second, observed in INDEPENDENCE_QUESTIONS:
    connected = clinical_network.is_dconnected(first, second, observed=observed)
    truth = "dependent" if connected else "independent"
    prediction = MY_PREDICTIONS[number]
    if prediction == truth:
        correct += 1
    mark = "" if prediction == truth else "  <--"
    observed_text = ", ".join(observed) if observed else "(nothing)"
    print(f"  {number:>2}  {first + ' / ' + second:<32}  {observed_text:<16}"
          f"  {prediction:<12}  {truth:<12}{mark}")

print(f"\n{correct} of {len(INDEPENDENCE_QUESTIONS)} match the graph.")

Questions 3 and 4 are the pair that matters: the same two variables, independent with nothing
observed and dependent once you observe `dyspnea`. Question 7 is the same effect reaching further —
observing dyspnea links heart failure to lung cancer, and lung cancer already reaches the chest
X-ray, so a heart failure diagnosis changes what you expect the film to show. Questions 2 and 6
are the ordinary case, where observing the middle of a chain or the common cause of a fork cuts
the connection.

One caution before we leave this. d-separation is a claim about the *graph*, not about the world.
If you drew the arrows wrong, pgmpy will confidently report independences that do not hold in any
patient.

## Part 3 — A protocol as a finite-state machine

Second half. A deterministic FSM is a set of states, a set of inputs, and a transition function
saying where each (state, input) pair leads — the "Finite-state machines: Formal definition"
slide. In Python that function is a dict. Our machine is a heparin infusion titration protocol
driven by the six-hourly aPTT result, and the reason it needs *states* rather than *rules* is that
the same lab value means different things depending on where the patient already is: an aPTT above
range while running at a therapeutic rate means reduce the rate, while an aPTT above range when
you have already reduced once means stop and hold.

In [ ]:
# (current state, input) -> (next state, the order that gets written)
HEPARIN_PROTOCOL = {
    ("STARTING", "aptt_low"):                ("SUBTHERAPEUTIC",   "increase rate by 2 units/kg/hr"),
    ("STARTING", "aptt_therapeutic"):        ("THERAPEUTIC",      "continue current rate"),
    ("STARTING", "aptt_high"):               ("SUPRATHERAPEUTIC", "reduce rate by 2 units/kg/hr"),

    ("SUBTHERAPEUTIC", "aptt_low"):          ("SUBTHERAPEUTIC",   "increase rate by 2 units/kg/hr again"),
    ("SUBTHERAPEUTIC", "aptt_therapeutic"):  ("THERAPEUTIC",      "continue current rate"),
    ("SUBTHERAPEUTIC", "aptt_high"):         ("SUPRATHERAPEUTIC", "reduce rate by 2 units/kg/hr"),

    ("THERAPEUTIC", "aptt_low"):             ("SUBTHERAPEUTIC",   "increase rate by 1 unit/kg/hr"),
    ("THERAPEUTIC", "aptt_therapeutic"):     ("THERAPEUTIC",      "continue current rate"),
    ("THERAPEUTIC", "aptt_high"):            ("SUPRATHERAPEUTIC", "reduce rate by 2 units/kg/hr"),

    # The same input, one state further along, means something else entirely.
    ("SUPRATHERAPEUTIC", "aptt_low"):        ("THERAPEUTIC",      "resume previous rate"),
    ("SUPRATHERAPEUTIC", "aptt_therapeutic"):("THERAPEUTIC",      "continue reduced rate"),
    ("SUPRATHERAPEUTIC", "aptt_high"):       ("HELD",             "HOLD infusion for 1 hour"),

    ("HELD", "aptt_low"):                    ("SUBTHERAPEUTIC",   "restart at 50% of previous rate"),
    ("HELD", "aptt_therapeutic"):            ("THERAPEUTIC",      "restart at 75% of previous rate"),
    ("HELD", "aptt_high"):                   ("HELD",             "keep holding, recheck in 1 hour"),

    ("STOPPED", "aptt_low"):                 ("STOPPED",          "infusion stopped; no titration"),
    ("STOPPED", "aptt_therapeutic"):         ("STOPPED",          "infusion stopped; no titration"),
    ("STOPPED", "aptt_high"):                ("STOPPED",          "infusion stopped; no titration"),
}

PROTOCOL_STATES = ["STARTING", "SUBTHERAPEUTIC", "THERAPEUTIC", "SUPRATHERAPEUTIC", "HELD", "STOPPED"]

# TODO: the protocol above has no answer for a patient who starts bleeding. Below this comment,
#       add HEPARIN_PROTOCOL[(state, "bleeding")] = ("STOPPED", "stop infusion, stat CBC,
#       consider protamine") for every state in PROTOCOL_STATES. A three-line loop does it.


def run_protocol(transitions, inputs, start_state="STARTING"):
    """Feed inputs to the machine one at a time and print the trace."""
    state = start_state
    print(f"  {'step':>4}  {'input':<18}  {'state before':<18}  {'state after':<18}  order")
    for step_number, symbol in enumerate(inputs, start=1):
        if (state, symbol) in transitions:
            next_state, order = transitions[(state, symbol)]
        else:
            # An FSM is only defined on the transitions you wrote down. This one is not total.
            next_state, order = state, "NO TRANSITION DEFINED -- protocol does not cover this"
        print(f"  {step_number:>4}  {symbol:<18}  {state:<18}  {next_state:<18}  {order}")
        state = next_state
    return state


six_hourly_aptt = ["aptt_low", "aptt_therapeutic", "aptt_high", "aptt_high", "aptt_therapeutic"]
final_state = run_protocol(HEPARIN_PROTOCOL, six_hourly_aptt)
print(f"\nfinal state: {final_state}")

Look at steps 3 and 4. The identical input, `aptt_high`, first reduces the rate and then holds
the infusion, because the machine remembered that it had already reduced once. A rule base keyed
only on the current aPTT cannot express that without smuggling the history in as extra facts, which
is precisely what a state is.

Now the failure. Run the next cell, in which the patient develops melena partway through. The
protocol has no transition for `bleeding` until you add one, and an undefined transition is not a
crash — it is a machine that silently keeps going in the state it was already in.

In [ ]:
bleeding_course = ["aptt_low", "aptt_therapeutic", "bleeding", "aptt_high", "aptt_therapeutic"]
final_state = run_protocol(HEPARIN_PROTOCOL, bleeding_course)
print(f"\nfinal state: {final_state}")
print()
if ("THERAPEUTIC", "bleeding") in HEPARIN_PROTOCOL:
    print("With the bleeding transitions added, the machine absorbs into STOPPED and stays there,")
    print("and the two later aPTT results correctly change nothing.")
else:
    print("Without the bleeding transitions, the machine ignored the bleed and carried on titrating")
    print("heparin for two more cycles. Nothing errored. Add the transitions and rerun both cells.")

## Part 4 — Conway's Game of Life, which is also a state machine

The deck makes the point that Life is an FSM: every cell is a two-state machine — alive or dead —
whose input is the number of live neighbours, and whose transition table is four lines long. A live
cell with two or three live neighbours stays alive; anything else dies. A dead cell with exactly
three live neighbours comes alive. There is no other rule, no global coordinator, and nothing that
knows what a glider is.

In [ ]:
GRID_SIZE = 12


def empty_grid(size):
    return [[0] * size for _ in range(size)]


def live_neighbours(grid, row, column):
    """Count the live cells in the eight surrounding positions, staying inside the grid."""
    total = 0
    for row_offset in (-1, 0, 1):
        for column_offset in (-1, 0, 1):
            if row_offset == 0 and column_offset == 0:
                continue
            neighbour_row, neighbour_column = row + row_offset, column + column_offset
            if 0 <= neighbour_row < len(grid) and 0 <= neighbour_column < len(grid):
                total += grid[neighbour_row][neighbour_column]
    return total


def next_generation(grid):
    new_grid = empty_grid(len(grid))
    for row in range(len(grid)):
        for column in range(len(grid)):
            neighbours = live_neighbours(grid, row, column)
            if grid[row][column] == 1:
                new_grid[row][column] = 1 if neighbours in (2, 3) else 0
            else:
                new_grid[row][column] = 1 if neighbours == 3 else 0
    return new_grid


def show_grid(grid, label):
    print(label)
    for row in grid:
        print("  " + "".join("#" if cell else "." for cell in row))
    print()


grid = empty_grid(GRID_SIZE)
for row, column in [(1, 2), (2, 3), (3, 1), (3, 2), (3, 3)]:    # a glider
    grid[row][column] = 1

show_grid(grid, "generation 0")

### Predict before you run

That five-cell shape is a glider. Before running the next cell, commit to two answers.

1. After 12 generations, is the shape still there, gone, or turned into something else?
2. If it is still there, where is it? Give a row and column offset from where it started.

The transition rule says nothing about motion, direction, or shape. Anything that looks like
movement has to be an artifact of five cells repeatedly killing and creating each other.

In [ ]:
for generation in range(1, 13):
    grid = next_generation(grid)
    if generation in (4, 8, 12):
        show_grid(grid, f"generation {generation}")

print("The glider is translated one row down and one column right every four generations,")
print("and it is not the same five cells: every one of them has died and been replaced.")

## Talk about it

1. Explaining away is what makes a clinician stop the workup once one diagnosis is confirmed. In
   our network, confirming lung cancer dropped heart failure from 51% to 13%. When is that
   reasoning right, and when is it the mechanism behind a missed second diagnosis?

2. Our CPD for dyspnea has eight numbers, and none of them came from data. If you were building
   this for real, where would each number come from, and what would you do about the combinations
   that almost never occur — COPD and lung cancer and heart failure all at once?

3. The heparin machine ignored a gastrointestinal bleed and kept titrating, without erroring,
   because we had not written that transition. Rule bases fail loudly when they cannot cover a
   case, since no rule fires; state machines fail quietly, since the state simply does not change.
   Which failure mode would you rather ship into a hospital, and how would you test for the one
   you chose?

## Solutions

Completed versions of the two TODOs. These are markdown, not code cells.

**Part 2 — the answers d-separation gives**

```python
MY_PREDICTIONS = {
    1: "dependent",     # chain smoking -> lung_cancer -> chest_xray, nothing blocking it
    2: "independent",   # observing the middle of a chain blocks it
    3: "independent",   # no edge, no shared parent: a collider at dyspnea, and it is unobserved
    4: "dependent",     # observing the collider opens the path. This is explaining away.
    5: "dependent",     # a fork at smoking, unobserved, so the two children are dependent
    6: "independent",   # observing the common cause blocks the fork
    7: "dependent",     # dyspnea opens the collider, then lung_cancer -> chest_xray carries on
}
```

**Part 3 — bleeding stops the infusion from any state**

```python
for state in PROTOCOL_STATES:
    HEPARIN_PROTOCOL[(state, "bleeding")] = ("STOPPED",
                                             "stop infusion, stat CBC, consider protamine")
```

Run that once, then rerun the two protocol cells. `STOPPED` is an absorbing state, so the two aPTT
results that arrive after the bleed correctly produce no titration at all.

**Optional — is the protocol total?** A machine that has an answer for every state and every input
cannot fall through to the silent default. Checking that is four lines, and it is the kind of test
worth writing before a protocol goes anywhere near a patient:

```python
PROTOCOL_INPUTS = ["aptt_low", "aptt_therapeutic", "aptt_high", "bleeding"]

missing = [(state, symbol) for state in PROTOCOL_STATES for symbol in PROTOCOL_INPUTS
           if (state, symbol) not in HEPARIN_PROTOCOL]
print(f"{len(missing)} undefined transitions: {missing}")
```
